# Stage 2 TEST, second evaluation: end to end on component-level omissions (IE643 floor-plan completion)

Every typed CubiCasa5K **test** drawing gets one single-component omission (door drawn over, door leaf erased, window,
wall piece, railing, fixture, a room's walls; kinds rotated) and runs through the whole frozen pipeline
(`src/stage2/test_components.py`). Same inputs, offline installs and two-GPU sharding as the final-test notebook;
every finished case is saved at once (resume by re-running with this notebook's output attached).
`SMOKE = True`: the first 3 **val** drawings only (code check). Output: `/kaggle/working/test_components/`
(`test_components.md` + per-case jsonl) and `test_components.zip`.

In [ ]:
import glob, os, shutil, subprocess, sys, time, zipfile
SMOKE = False          # True: 2 cases per section on the VAL split (code check); False: the real test, run once
WORK = "/kaggle/working"
CODE = "/tmp/code"
OUT = f"{WORK}/test_components" + ("_smoke_val" if SMOKE else "")
os.makedirs(OUT, exist_ok=True)

def find(pattern):
    """Files in the attached inputs (and in /tmp/s2final, where zipped input folders are unpacked)."""
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
                  + glob.glob(f"/tmp/s2final/**/{pattern}", recursive=True))

# 1. code (the floorplan-code dataset holds code.zip, or its unpacked folder)
hits = [h for h in find("src/utils.py") if "floorplan-code" in h]
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE, dirs_exist_ok=True)
else:
    zipfile.ZipFile([h for h in find("code.zip") if "floorplan-code" in h][0]).extractall(CODE)
print("code ->", CODE, os.listdir(CODE)[:10])

In [ ]:
# 1b. packages, offline from the wheels in floorplan-stage2-final, then a version check
# floorplan-stage2-final was uploaded with its folders zipped (kaggle datasets create -r zip); Kaggle normally unpacks
# them, but if a folder stayed a zip, unpack it here
for folder in ("wheels", "checkpoints"):
    if not [h for h in glob.glob(f"/kaggle/input/**/{folder}", recursive=True) if os.path.isdir(h)]:
        for z in find(f"{folder}.zip"):
            zipfile.ZipFile(z).extractall(f"/tmp/s2final/{folder}")
            print("unpacked", z)
wheels = sorted({os.path.dirname(w) for w in find("wheels/*.whl")})
need = ["segmentation-models-pytorch==0.3.4", "timm==0.9.7", "pretrainedmodels==0.7.4", "efficientnet-pytorch==0.7.1"]
try:
    import diffusers  # noqa: F401  (Kaggle's own diffusers is fine when present)
except ImportError:
    need.append("diffusers==0.30.3")
args = [sys.executable, "-m", "pip", "install", "-q", *need]
if wheels:
    args += ["--no-index", "--find-links", wheels[0]]
print(" ".join(args)); subprocess.run(args, check=True)
import importlib.metadata as md
versions = {p: md.version(p) for p in ("segmentation-models-pytorch", "timm", "diffusers", "torch")}
print(versions)
assert versions["segmentation-models-pytorch"] == "0.3.4" and versions["timm"] == "0.9.7", versions

In [ ]:
# 2. checkpoints: floorplan-stage2-final/checkpoints/<path> -> CODE/<path> (paths of configs/stage2/pipeline.yaml)
import yaml
paths = yaml.safe_load(open(f"{CODE}/configs/stage2/pipeline.yaml", encoding="utf-8"))["checkpoints"]
roots = [h[: -len("checkpoints/" + p)] for name, p in paths.items() for h in find("checkpoints/" + p)]
assert roots, "floorplan-stage2-final (checkpoints) is not attached"
root = roots[0]
for name, p in paths.items():
    src = os.path.join(root, "checkpoints", p)
    if os.path.exists(src):
        os.makedirs(os.path.dirname(f"{CODE}/{p}"), exist_ok=True)
        shutil.copy(src, f"{CODE}/{p}")
        print(f"{name:18s} ok")
    else:
        print(f"{name:18s} MISSING", src)

# 3. test data
split = "val" if SMOKE else "test"
PZ, CZ = find(f"parser_{split}.zip"), ["unused"]
assert PZ, f"parser_{split}.zip not found: attach the floorplan-stage2-data output"
PZ, CZ = PZ[0], CZ[0]
print(PZ, CZ)

In [ ]:
# 4a. resume: copy the finished cases of an earlier version of this notebook (if its output is attached)
for f in find("test_components/part*.jsonl"):
    dst = f"{OUT}/{os.path.basename(f)}"
    if not os.path.exists(dst):
        shutil.copy(f, dst)
        print("resuming from", f, sum(1 for _ in open(f)), "cases")

# 4b. quick GPU check (no test data touched)
import torch
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 1, "set Accelerator = GPU T4 x2"

In [ ]:
# 4c. the evaluation: one process per GPU
n = max(1, torch.cuda.device_count())
threads = str(max(1, (os.cpu_count() or 2) // n))   # the CPU parts (CHECK, verifier) of both processes share the cores
env = {**os.environ, "PYTHONUTF8": "1", "PYTHONUNBUFFERED": "1", "STAGE2_DEVICE": "cuda", "PYTHONPATH": CODE,
       "OMP_NUM_THREADS": threads, "MKL_NUM_THREADS": threads}
procs = []
for k in range(n):
    log = open(f"{OUT}/part{k}.log", "a")
    cmd = [sys.executable, "-m", "src.stage2.test_components", "--parser-zip", PZ, "--out", OUT,
           "--part", str(k), "--parts", str(n)] + (["--smoke"] if SMOKE else [])
    procs.append(subprocess.Popen(cmd, cwd=CODE, stdout=log, stderr=subprocess.STDOUT,
                                  env={**env, "CUDA_VISIBLE_DEVICES": str(k)}))
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(120)
    done = sum(sum(1 for _ in open(f)) for f in glob.glob(f"{OUT}/part*.jsonl"))
    print(f"{(time.time() - t0) / 3600:.2f} h, {done} cases done", flush=True)
codes = [p.wait() for p in procs]
print("exit codes", codes)
for k in range(n):
    print(f"===== part{k}.log (tail)"); print("".join(open(f"{OUT}/part{k}.log").readlines()[-20:]))
if any(codes):
    print("!!!!! A SHARD FAILED (exit code != 0): the report below is INCOMPLETE. Fix, then re-run this notebook with "
          "its own output attached: finished cases are kept.")

In [ ]:
# 5. report + zip of all outputs
subprocess.run([sys.executable, "-m", "src.stage2.test_components", "--report", "--out", OUT], cwd=CODE,
               env={**env, "STAGE2_DEVICE": "cpu"}, check=True)
shutil.make_archive(f"{WORK}/test_components", "zip", OUT)
print(open(f"{OUT}/test_components.md", encoding="utf-8").read())